## >>> VERSION: 2026-07-24  ·  v7-batch  <<<
**How to use this version:** run it **once per batch dataset** (`dataset_batch1`, `dataset_batch2`, `dataset_batch3`). Attach only ONE batch at a time (+ code, checkpoint, scripts) and detach the others. Each batch has ~167 images and finishes in ~2.7h — under Kaggle's ~3.3h kernel recycle, so no run hits the wall. The `(expected 500)` text it prints is cosmetic; it will show ~167 for a batch, that's fine. After each run, download `metrics.zip` + `dit_outputs.zip` and **label them batch1 / batch2 / batch3** (don't overwrite). The 3 batches get combined afterward into the final 500-row result.

*No cell changes are needed — the notebook auto-detects whichever batch folder (Drop/ + Clear/) is attached. If the top of your uploaded notebook does NOT say `v7-batch`, re-upload.*

---

# Stage 2 - DiT alone (batched) on the 500-image benchmark (INSTRUMENTED)

Runs **no-restoration** then **DiT alone** on the attached batch. Writes per-image metrics,
a summary, and resource stats to files; saves DiT output images; zips everything.
Eval runs via the `!` shell magic (no duplication) and skips already-done images (resume-safe).

All metrics at a fixed **256x256**. **Inputs:** RaindropClarity code, ONE `dataset_batchN`,
`RainDrop_DiT_ddpm.pth.tar`, and `scripts/` (score_pairs.py, model_stats.py). **GPU T4 x2, Internet on.**

In [ ]:
import shutil, os, glob
# clear any stale DiT lock ONCE at the very start of the run (nothing else resets it,
# so the lock stays valid for the whole run and any duplicate eval process will hit it)
if os.path.exists('/kaggle/working/dit_eval.lock'):
    os.remove('/kaggle/working/dit_eval.lock')
src = None
for d, _, files in os.walk('/kaggle/input'):
    if 'eval_diffusion_day_dit.py' in files:
        src = d; break
assert src, 'Could not find the RaindropClarity code under /kaggle/input'
shutil.copytree(src, '/kaggle/working/RaindropClarity', dirs_exist_ok=True)
os.chdir('/kaggle/working/RaindropClarity')
print('cwd =', os.getcwd())
for s in glob.glob('/kaggle/input/**/score_pairs.py', recursive=True):
    shutil.copy(s, 'score_pairs.py'); print('got', s)
for s in glob.glob('/kaggle/input/**/model_stats.py', recursive=True):
    shutil.copy(s, 'model_stats.py'); print('got', s)
os.makedirs('/kaggle/working/metrics', exist_ok=True)

In [ ]:
!pip install einops lpips -q

In [ ]:
# locate the 500-image dataset (folder with BOTH Drop/ and Clear/) and the DiT checkpoint
data_dir = None
for d, subs, _ in os.walk('/kaggle/input'):
    if 'Drop' in subs and 'Clear' in subs:
        data_dir = d; break
assert data_dir, "Couldn't find Drop/ and Clear/ under /kaggle/input"
ckpts = glob.glob('/kaggle/input/**/RainDrop_DiT_ddpm.pth.tar', recursive=True)
assert ckpts, "Couldn't find RainDrop_DiT_ddpm.pth.tar under /kaggle/input"
os.environ['DATA_DIR'] = data_dir
os.environ['CKPT'] = ckpts[0]
n_imgs = len(glob.glob(os.path.join(data_dir,'Drop','**','*.png'), recursive=True))
print('DATA_DIR =', data_dir)
print('CKPT     =', ckpts[0])
print('images   =', n_imgs, '(expected 500)')

## Config 1 - No restoration (no GPU) + plumbing smoke test

In [ ]:
# smoke test first: 20 images, confirms score_pairs + CSV + summary work
!python score_pairs.py --pred "$DATA_DIR/Drop" --gt "$DATA_DIR/Clear" \
    --name _smoketest --out_dir /kaggle/working/metrics --limit 20
import pandas as pd
df = pd.read_csv('/kaggle/working/metrics/_smoketest_per_image.csv')
print('\nsmoke test rows:', len(df), '(expect 20)')
assert len(df) == 20, 'smoke test did not produce 20 rows - fix before continuing'
print('PLUMBING OK')

In [ ]:
# full no-restoration result (all 500)
!python score_pairs.py --pred "$DATA_DIR/Drop" --gt "$DATA_DIR/Clear" \
    --name no_restoration --out_dir /kaggle/working/metrics
print(open('/kaggle/working/metrics/no_restoration_summary.txt').read())

## Config 2 - DiT alone (the ~9h run)

In [ ]:
# static resource stats: parameter count + checkpoint size (no GPU, instant)
!python model_stats.py --ckpt "$CKPT" --name DiT \
    --out_csv /kaggle/working/metrics/resource_static.csv
print(open('/kaggle/working/metrics/resource_static.csv').read())

In [ ]:
# point DiT's config at the 500 dataset (EXACTLY like the working 150 notebook: num_workers 2)
import re, pathlib
cfg = pathlib.Path('configs/daytime_64.yml')
t = cfg.read_text()
t = re.sub(r'data_dir:.*', f'data_dir: "{os.environ["DATA_DIR"]}"', t)
t = re.sub(r'num_workers:.*', 'num_workers: 2', t)
cfg.write_text(t)
import torch; print('GPU on:', torch.cuda.is_available(), ' device_count:', torch.cuda.device_count())

In [ ]:
# (1) PID print in main() to confirm single execution (no other change to how it runs).
import pathlib
_p = pathlib.Path('eval_diffusion_day_dit.py')
_src = _p.read_text()
if 'DIT_EVAL_PID' not in _src:
    _src = _src.replace(
        'def main():\n',
        "def main():\n    import os as _o\n    print('DIT_EVAL_PID', _o.getpid(), flush=True)\n", 1)
    _p.write_text(_src)
    print('added PID print to main()')
else:
    print('PID print already present')

# (2) RESUME: skip any image whose output PNG already exists. If Kaggle recycles the kernel
#     and re-runs the notebook from the top, the eval CONTINUES from where it stopped
#     instead of redoing 00002 - so it finishes across however many restarts happen.
_r = pathlib.Path('models/restoration.py')
_rs = _r.read_text()
if '_RESUME_SKIP' not in _rs:
    _rs = _rs.replace(
        "                print(datasetname, id, frame)\n",
        "                print(datasetname, id, frame)\n"
        "                _op = os.path.join(image_folder, datasetname, 'output', id, frame + '.png')  # _RESUME_SKIP\n"
        "                if os.path.exists(_op):\n"
        "                    print('  (skip, already done)', flush=True); continue\n", 1)
    _r.write_text(_rs)
    print('added RESUME skip-if-exists to restoration.py')
else:
    print('resume skip already present')

In [ ]:
# DiT eval via the ! shell magic (bypasses the debugger). Counts outputs before/after so
# per-image timing is correct even if the run was resumed after a Kaggle restart.
import subprocess, time, os, glob
def _count_outputs():
    o = sorted(glob.glob('results/RainDrop/Raindrop_DiT/*/output'), key=os.path.getmtime, reverse=True)
    return len(glob.glob(o[0] + '/**/*.png', recursive=True)) if o else 0
n_before = _count_outputs()
print('outputs already present (from earlier attempt):', n_before)
mem_log = '/kaggle/working/metrics/gpu_mem_dit.csv'
logf = open(mem_log, 'w')
poller = subprocess.Popen(
    ['nvidia-smi','--query-gpu=memory.used','--format=csv,noheader,nounits','-l','1'],
    stdout=logf)
t0 = time.time()
rc = get_ipython().system(
    'CUDA_VISIBLE_DEVICES=0 python eval_diffusion_day_dit.py '
    '--config daytime_64.yml --test_set Raindrop_DiT --resume "$CKPT"')
elapsed = time.time() - t0
poller.terminate(); logf.close()
n_after = _count_outputs()
print('eval done. elapsed:', round(elapsed,1), 's | this session:', n_after - n_before, '| total:', n_after)
os.environ['DIT_ELAPSED'] = str(elapsed)
os.environ['DIT_IMGS_THIS_SESSION'] = str(max(n_after - n_before, 1))
os.environ['DIT_N_TOTAL'] = str(n_after)

In [ ]:
# SALVAGE: zip whatever DiT outputs exist NOW (before scoring), so an interrupted run
# still leaves downloadable restored images.
import glob, os, shutil
_outs = sorted(glob.glob('results/RainDrop/Raindrop_DiT/*/output'), key=os.path.getmtime, reverse=True)
if _outs:
    _o = _outs[0]
    _n = len(glob.glob(_o + '/**/*.png', recursive=True))
    shutil.copytree(_o, '/kaggle/working/dit_outputs', dirs_exist_ok=True)
    os.system('cd /kaggle/working && rm -f dit_outputs.zip && zip -r dit_outputs.zip dit_outputs -q')
    print(f'SALVAGE: {_n} DiT output images -> /kaggle/working/dit_outputs.zip (safe to download now)')
else:
    print('SALVAGE: no DiT outputs found - eval produced nothing')

In [ ]:
# runtime resource stats — RESTART-PROOF timing.
# per-image time is measured from THIS session's actual work (elapsed / images_this_session),
# so it's correct even if the run was resumed after a restart. Total time is projected to 500.
import csv as _csv, os
elapsed = float(os.environ['DIT_ELAPSED'])
imgs_session = float(os.environ.get('DIT_IMGS_THIS_SESSION', '1'))
N_total = int(os.environ.get('DIT_N_TOTAL', '0'))
mem_vals = []
for line in open('/kaggle/working/metrics/gpu_mem_dit.csv'):
    line = line.strip()
    if line.isdigit(): mem_vals.append(int(line))
peak_mem = max(mem_vals) if mem_vals else -1
per_image = elapsed / imgs_session
row = {'model':'DiT',
       'per_image_s': round(per_image, 4),
       'throughput_img_per_s': round(1.0/per_image, 4),
       'total_time_s_full500': round(per_image*500, 2),
       'peak_gpu_mem_MB': peak_mem,
       'n_images_total': N_total,
       'n_images_this_session': int(imgs_session)}
rt = '/kaggle/working/metrics/resource_runtime.csv'
with open(rt,'w',newline='') as f:   # overwrite so there's ONE clean row
    w=_csv.DictWriter(f,fieldnames=list(row.keys()))
    w.writeheader(); w.writerow(row)
print(row)
print(open(rt).read())

In [ ]:
# score DiT outputs at 256x256 -> per-image CSV + summary (partial-tolerant: scores whatever exists)
import glob, os
_outs = sorted(glob.glob('results/RainDrop/Raindrop_DiT/*/output'), key=os.path.getmtime, reverse=True)
if not _outs:
    print('NO DiT OUTPUTS to score - eval produced nothing this run')
else:
    base = _outs[0][:-len('output')]
    os.environ['DIT_OUT'] = base + 'output'
    os.environ['DIT_GT']  = base + 'gt'
    _n = len(glob.glob(os.environ['DIT_OUT'] + '/**/*.png', recursive=True))
    print('scoring', _n, 'DiT outputs (500 = full run; fewer = partial)')
    os.system('python score_pairs.py --pred "%s" --gt "%s" --name dit_alone --out_dir /kaggle/working/metrics'
              % (os.environ['DIT_OUT'], os.environ['DIT_GT']))
    print(open('/kaggle/working/metrics/dit_alone_summary.txt').read())

## Package + verify

In [ ]:
# package metrics; refresh dit_outputs.zip in case more images landed since SALVAGE
import os, glob, shutil
os.system('cd /kaggle/working && rm -f metrics.zip && zip -r metrics.zip metrics -q')
_outs = sorted(glob.glob('results/RainDrop/Raindrop_DiT/*/output'), key=os.path.getmtime, reverse=True)
if _outs:
    shutil.copytree(_outs[0], '/kaggle/working/dit_outputs', dirs_exist_ok=True)
    os.system('cd /kaggle/working && rm -f dit_outputs.zip && zip -r dit_outputs.zip dit_outputs -q')
print('wrote metrics.zip and dit_outputs.zip')
print('DOWNLOAD from Output panel: metrics.zip (CSVs + stats), dit_outputs.zip (restored images)')

In [ ]:
# final verification - report what exists and row counts (partial-tolerant, never crashes)
import os
try:
    import pandas as pd
except Exception:
    pd = None
m = '/kaggle/working/metrics'
def chk(p, expect=None):
    ok = os.path.exists(p); extra=''
    if ok and p.endswith('.csv') and pd is not None:
        try:
            n = len(pd.read_csv(p)); extra=' (%d rows)' % n
            if expect and n != expect: extra += '  << expected %d (partial run)' % expect
        except Exception as e:
            extra = ' (unreadable: %s)' % e
    print(('OK   ' if ok else 'MISS ') + p + extra)
chk(m+'/no_restoration_per_image.csv', 500)
chk(m+'/no_restoration_summary.txt')
chk(m+'/dit_alone_per_image.csv', 500)
chk(m+'/dit_alone_summary.txt')
chk(m+'/resource_static.csv')
chk(m+'/resource_runtime.csv')
print('\ndit_alone rows = DiT images actually scored (500 = full, fewer = partial but usable).')
print('Near the DiT start, exactly ONE "DIT_EVAL_PID" line = no duplication.')
print('Any "DUPLICATE_DIT_PROCESS" line = the guard caught and killed a 2nd process.')